<a href="https://colab.research.google.com/github/sreent/machine-learning/blob/main/Lectures/4%20Linear%20Regression/Lab%3A%20Linear%20Regression%20from%20Scratch.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Linear Regression from Scratch

### **Objective**
In this lab, we’ll implement linear regression step-by-step in Python. We’ll cover essential tasks, such as handling outliers, scaling features, building a `scikit-learn`-compatible model, and evaluating its performance. Each section is explained in detail to make it beginner-friendly.

### **Lab Outline**

1. Introduction to Linear Regression
2. Data Generation and Initial Exploration
3. Handling Outliers with the IQR Rule
4. Feature Scaling with `StandardScaler`
5. Implementing Linear Regression from Scratch
6. Evaluating the Model
7. Comparison with `scikit-learn`’s LinearRegression
8. Optional Exploration: Overfitting and Underfitting

### **1. Introduction to Linear Regression**

**Linear regression** is a basic machine learning algorithm used to model the relationship between input features (independent variables) and a target variable (dependent variable). The goal is to fit a line (or hyperplane) through the data points that best represents this relationship.

#### Real-World Analogy
Think of linear regression like predicting house prices based on square footage. We can observe a general trend: larger houses tend to be more expensive. Linear regression helps us quantify this relationship, so we can predict house prices based on square footage and other factors.

#### Mathematical Model
The linear model can be written as:
$$
y = X \vec{w} + \epsilon
$$
where:
- $y$ is the predicted target variable,
- $X$ is the feature matrix (input variables),
- $\vec{w}$ represents the weights or coefficients for each feature,
- $\epsilon$ is the error term, representing the difference between actual and predicted values.

### **2. Data Generation and Initial Exploration**

Let’s start by generating synthetic data, including some random noise to simulate real-world data. We’ll add a few extreme values (outliers) to observe their effect on the model.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

# Generate synthetic data for testing
np.random.seed(0)
X = 2 * np.random.rand(100, 1)
y = 4 + 3 * X + np.random.randn(100, 1)

# Introduce outliers
X_outliers = np.array([[1.5], [2.0], [2.5]])
y_outliers = np.array([[10], [12], [15]])
X = np.vstack([X, X_outliers])
y = np.vstack([y, y_outliers])

# Plot the data with outliers
plt.scatter(X, y, color='blue', label="Data with Outliers")
plt.xlabel("x")
plt.ylabel("y")
plt.title("Scatter Plot of Data with Outliers")
plt.legend()
plt.grid()
plt.show()

### **3. Handling Outliers with the IQR Rule**

Outliers are extreme values that can distort the regression line, pulling it toward them and resulting in biased estimates. To manage this, we’ll use the **1.5x IQR (Interquartile Range) rule** to identify and remove outliers.

#### Explanation of IQR Rule
- **Step 1**: Calculate the first quartile (Q1) and third quartile (Q3) for the target variable $y$.
- **Step 2**: Calculate the Interquartile Range (IQR), which is $Q3 - Q1$.
- **Step 3**: Define the lower and upper bounds for outliers as $Q1 - 1.5 \times \text{IQR}$ and $Q3 + 1.5 \times \text{IQR}$, respectively.
- **Step 4**: Filter out values outside these bounds.

In [ ]:
import pandas as pd

# Define a function to remove outliers using the 1.5x IQR rule
def remove_outliers_iqr(X, y):
    # Convert y to a DataFrame for easy manipulation with pandas
    y_df = pd.DataFrame(y, columns=['target'])

    # Step 1: Calculate Q1 (25th percentile) and Q3 (75th percentile) for y
    Q1 = y_df['target'].quantile(0.25)
    Q3 = y_df['target'].quantile(0.75)

    # Step 2: Calculate the Interquartile Range (IQR)
    IQR = Q3 - Q1

    # Step 3: Define the lower and upper bounds for outliers
    lower_bound = Q1 - 1.5 * IQR
    upper_bound = Q3 + 1.5 * IQR

    # Step 4: Filter out the outliers
    mask = (y_df['target'] >= lower_bound) & (y_df['target'] <= upper_bound)
    X_filtered = X[mask.values]  # mask.values to apply filter to numpy array X
    y_filtered = y[mask.values]

    return X_filtered, y_filtered

# Remove outliers and plot the updated data
X, y = remove_outliers_iqr(X, y)
plt.scatter(X, y, color='green', label="Data without Outliers")
plt.xlabel("x")
plt.ylabel("y")
plt.title("Scatter Plot after Removing Outliers")
plt.legend()
plt.show()

### **4. Feature Scaling with `StandardScaler`**

Feature scaling ensures that all features are on a comparable scale, which helps the model converge faster and makes it more interpretable. We’ll use `StandardScaler` from `scikit-learn` to standardize the features, so they have a mean of 0 and a standard deviation of 1.

In [ ]:
from sklearn.preprocessing import StandardScaler

# Standardize features using StandardScaler
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)

### **5. Implementing Linear Regression from Scratch**

Now, we’ll create a custom `MyLinearRegressor` class, following `scikit-learn` conventions, and implementing the **Normal Equation** for solving linear regression directly. This involves calculating weights using matrix operations.

#### Explanation of the Class Methods
- **`fit` Method**: Calculates the weights using the Normal Equation. This approach doesn’t require iterative optimization.
- **`predict` Method**: Generates predictions using these weights.
- **`score` Method**: Calculates the $R^2$ score, a common metric to evaluate model fit.

In [ ]:
from sklearn.base import BaseEstimator, RegressorMixin

class MyLinearRegressor(BaseEstimator, RegressorMixin):
    def __init__(self):
        self.theta = None  # To store weights after training

    def fit(self, X, y):
        # Add a bias term (column of ones) to X
        X_b = np.c_[np.ones((X.shape[0], 1)), X]

        # Calculate theta using the normal equation:
        # theta = (X_b.T @ X_b)^-1 @ X_b.T @ y
        self.theta = np.linalg.inv(X_b.T @ X_b) @ X_b.T @ y
        return self

    def predict(self, X):
        # Add bias term to X and compute predictions
        X_b = np.c_[np.ones((X.shape[0], 1)), X]
        return X_b @ self.theta

    def score(self, X, y):
        # Calculate R^2 score
        y_pred = self.predict(X)
        u = ((y - y_pred) ** 2).sum()
        v = ((y - y.mean()) ** 2).sum()
        return 1 - u / v

**Note**: `BaseEstimator` and `RegressorMixin` make this model compatible with `scikit-learn` utilities, such as `cross_val_score`, which allows us to validate model performance more easily.


### **6. Evaluating the Model**

We’ll evaluate the model using **Mean Squared Error (MSE)** and \( R^2 \) score.

- **MSE**: Measures the average squared difference between actual and predicted values.
- **$R^2$**: Indicates the proportion of the variance in the target variable explained by the model.

In [ ]:
# Initialize and fit the model
model = MyLinearRegressor()
model.fit(X_scaled, y)

# Predict values and evaluate
y_pred = model.predict(X_scaled)
mse = np.mean((y - y_pred) ** 2)
print("Mean Squared Error:", mse)

r2_score = model.score(X_scaled, y)
print("R^2 Score:", r2_score)

### **7. Comparison with `scikit-learn`’s LinearRegression**

To validate our model, we’ll compare it with `scikit-learn`’s `LinearRegression` and use cross-validation to check its reliability.

In [ ]:
from sklearn.linear_model import LinearRegression
from sklearn.model_selection import cross_val_score

# Initialize and fit scikit-learn's LinearRegression
sklearn_model = LinearRegression()
scores = cross_val_score(sklearn_model, X_scaled, y, cv=5, scoring='r2')
print("Cross-validated R^2 scores (scikit-learn):", scores)
print("Mean R^2 score (scikit-learn):", scores.mean())


### **8. Optional Exploration: Overfitting and Underfitting**

In this optional section, we’ll experiment with **polynomial features** to explore overfitting. When the degree of the polynomial is too high, the model may start capturing noise in the data, resulting in **overfitting**—a model that performs well on the training data but poorly on new data.

#### Steps:
- Use `PolynomialFeatures` from `scikit-learn` to add polynomial terms to the features.
- Fit the model on these polynomial features and observe changes in Mean Squared Error (MSE) and $R^2$ score.

In [ ]:
from sklearn.preprocessing import PolynomialFeatures

# Generate polynomial features (e.g., degree=3)
poly = PolynomialFeatures(degree=3, include_bias=False)
X_poly = poly.fit_transform(X_scaled)

# Fit the model on polynomial features
model.fit(X_poly, y)
y_poly_pred = model.predict(X_poly)

# Evaluate the model with polynomial features
mse_poly = np.mean((y - y_poly_pred) ** 2)
r2_poly = model.score(X_poly, y)
print("Mean Squared Error (Polynomial Features):", mse_poly)
print("R^2 Score (Polynomial Features):", r2_poly)

#### Interpretation:
- **If MSE decreases** and **$R^2$ score increases**, the polynomial features have improved the model fit.
- **However, a very high degree** may lead to overfitting, where the model fits the training data too closely and fails to generalize to new data.

**Note**: Overfitting can often be managed by controlling the complexity of the model or using techniques such as regularization.


### **Reflection and Summary**

In this lab, you:
- Used the **1.5x IQR rule** to remove outliers, ensuring that extreme values didn’t skew the model.
- Standardized features using `scikit-learn`’s **StandardScaler** to improve model stability.
- Implemented **linear regression from scratch** using the Normal Equation.
- Evaluated model performance using **MSE** and **$R^2$** scores to measure fit and accuracy.
- Compared your custom implementation with `scikit-learn`’s **LinearRegression** to validate results.
- Explored the impact of adding polynomial features, illustrating **overfitting** and **underfitting**.

This lab covers essential steps in developing and evaluating a linear regression model from scratch, using best practices for data preprocessing and model validation. By completing this lab, you’ve gained a solid foundation in the practical application of linear regression.